# Phase 5: Atlas vs TimesFM and Chronos on the 2023 test and on live NOAA data, 2024 to now (ParadoxAI Lab)

Rules and predictions: `atlas/README.md`, "Phase 5", written before this ran. **Run:** Runtime → Change runtime type →
**A100** → Save → **Run all**. Results go to Drive `PWM-10M/small/h2h/`.

In [ ]:
# 1. Drive and GPU
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, json
W = '/content/drive/MyDrive/PWM-10M'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Code and the opponents' packages
!rm -rf /content/pwm && git clone -q --depth 1 -b atlas-v0.1 https://github.com/paradoxailab/paradox-world-model.git /content/pwm
!cd /content/pwm && git log --oneline -1
!pip install -q timesfm chronos-forecasting 2>&1 | tail -2

In [ ]:
# 3. WEATHER-5K from the Drive copy
R = '/content/raw'; os.makedirs(R, exist_ok=True)
if not os.path.exists(f'{R}/W5K_X.npy'):
    shutil.copy(f'{W}/data/W5K_META.npz', R); shutil.copy(f'{W}/data/W5K_X.npy', R)
!ls -la /content/raw

In [ ]:
# 4. Phase 5a: the 2023 test, once (same windows design as validation)
%cd /content/pwm/atlas/delhi
A10 = f'{W}/out/w5k_atlas_10m_h24_g1_s600.pt'
!ATLAS_OUT={W}/small PW_RAW=/content/raw python -u score_tsfm_w5k.py --year test --stations-every 8 --origin-every 48 --atlas atlas10m={A10},v2c=/content/pwm/models/series/57_atlas-5k_v2c_weather5k_adopted.pt --models persist,chronos-bolt,chronos2,chronos2mv,tfm25,tfm3 2>&1 | grep --line-buffered -v Warning
!ATLAS_OUT={W}/small python paired_w5k.py --year test --ours atlas10m --vs persist,chronos-bolt,chronos2,chronos2mv,tfm25,tfm3
!ATLAS_OUT={W}/small python paired_w5k.py --year test --ours v2c --vs persist,chronos-bolt,chronos2,chronos2mv,tfm25,tfm3

In [ ]:
# 5. Phase 5b data: fresh NOAA ISD-Lite observations for the same stations, 2024 to now
!python -u /content/pwm/atlas/delhi/tools/fetch_isd_live.py --meta /content/raw/W5K_META.npz --out /content/live --years 2024,2025,2026 | tee {W}/small/live_data_log.txt

In [ ]:
# 6. Phase 5b: the live test
!ATLAS_OUT={W}/small PW_RAW=/content/live python -u score_tsfm_w5k.py --year live --stations-every 8 --origin-every 96 --atlas atlas10m={A10},v2c=/content/pwm/models/series/57_atlas-5k_v2c_weather5k_adopted.pt --models persist,chronos-bolt,chronos2,chronos2mv,tfm25,tfm3 2>&1 | grep --line-buffered -v Warning
!ATLAS_OUT={W}/small python paired_w5k.py --year live --ours atlas10m --vs persist,chronos-bolt,chronos2,chronos2mv,tfm25,tfm3
!ATLAS_OUT={W}/small python paired_w5k.py --year live --ours v2c --vs persist,chronos-bolt,chronos2,chronos2mv,tfm25,tfm3